# Pemisahan Pada Dataset Final untuk Trainning Validation & Testing

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os
import re

print("=" * 75)
print("🚀 FASE 2: PEMBERSIHAN MUTLAK & STRATIFIED SPLIT 70/15/15")
print("=" * 75)

# 1. BACA FILE MASTER
input_csv = "../data/raw/dataset_politik_balanced.csv"
if not os.path.exists(input_csv):
    input_csv = "data/raw/dataset_politik_balanced.csv"

df_master = pd.read_csv(input_csv)
print(f"Total Master Dataset Awal    : {len(df_master):,} baris")

# -------------------------------------------------------------
# 2. SAPU BERSIH SELURUH KATA BOCOR DENGAN POLA YANG SAMA
# -------------------------------------------------------------
# Langsung cabut kata 'imposter', 'periksa fakta', dan 'faktanya' di mana pun berada
target_kebocoran = [
    r'\bimposter\b',
    r'\bhasil\s+periksa\s+fakta\b\s*:?',
    r'\bperiksa\s+fakta\b\s*:?',
    r'\bfaktanya\b\s*[:,]?'
]

print("--> Mencabut kata 'imposter', 'periksa fakta', dan 'faktanya'...")
for p in target_kebocoran:
    df_master['text'] = df_master['text'].str.replace(p, '', case=False, regex=True)

# Normalisasi spasi dan tanda titik
df_master['text'] = df_master['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
df_master['text'] = df_master['text'].apply(lambda x: x if str(x).endswith(('.', '!', '?')) else str(x) + ".")

# Filter minimal 10 kata agar tidak ada naskah kerdil
df_master['wc'] = df_master['text'].apply(lambda x: len(str(x).split()))
df_master = df_master[df_master['wc'] >= 10].drop(columns=['wc']).copy()

# Penyeimbangan Ulang 50:50 Presisi
min_c = min(df_master['label'].value_counts())
df_h = df_master[df_master['label'] == 1].sample(n=min_c, random_state=42)
df_f = df_master[df_master['label'] == 0].sample(n=min_c, random_state=42)

df_clean_master = pd.concat([df_h, df_f]).sample(frac=1, random_state=42).reset_index(drop=True)
df_clean_master['id'] = [f"POL-{i+1:05d}" for i in range(len(df_clean_master))]

# Simpan kembali master yang steril
df_clean_master.to_csv(input_csv, index=False, encoding='utf-8-sig')

# -------------------------------------------------------------
# 3. VERIFIKASI ULANG KEBOCORAN (PASTI 0)
# -------------------------------------------------------------
sisa_bocor = df_clean_master['text'].str.contains(r'\bimposter\b|\bperiksa\s+fakta\b|\bfaktanya\b', case=False, regex=True).sum()
print(f"Hasil Verifikasi Ulang Kebocoran : {sisa_bocor} baris bocor!")
assert sisa_bocor == 0, f"Masih ada {sisa_bocor} baris bocor!"
print("✅ STATUS: MASTER DATASET RESMI STERIL MUTLAK 100% (0 BOCOR)!")

# -------------------------------------------------------------
# 4. STRATIFIED SPLIT 70% TRAIN, 15% VAL, 15% TEST
# -------------------------------------------------------------
print("\n" + "-" * 75)
print("📦 MEMBAGI MENJADI 3 SUBSET RESMI (70% / 15% / 15%)...")

train_df, temp_df = train_test_split(
    df_clean_master, 
    test_size=0.30, 
    random_state=42, 
    stratify=df_clean_master['label']
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.50, 
    random_state=42, 
    stratify=temp_df['label']
)

# Simpan ke folder data/processed/
PROCESSED_DIR = "../data/processed" if os.path.exists("../data/processed") else "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_path = os.path.join(PROCESSED_DIR, "train.csv")
val_path   = os.path.join(PROCESSED_DIR, "val.csv")
test_path  = os.path.join(PROCESSED_DIR, "test.csv")

train_df.to_csv(train_path, index=False, encoding='utf-8-sig')
val_df.to_csv(val_path, index=False, encoding='utf-8-sig')
test_df.to_csv(test_path, index=False, encoding='utf-8-sig')

# =============================================================
# 5. LAPORAN HASIL AKHIR DI DISK
# =============================================================
print("=" * 75)
print(f"🏆 BERHASIL! TIGA FILE RESMI TERSIMPAN DI data/processed/ :")
print(f"• train.csv (70%) : {len(train_df):,} baris (Hoaks: {train_df['label'].sum():,} | Fakta: {len(train_df) - train_df['label'].sum():,})")
print(f"• val.csv   (15%) : {len(val_df):,} baris  (Hoaks: {val_df['label'].sum():,} | Fakta: {len(val_df) - val_df['label'].sum():,})")
print(f"• test.csv  (15%) : {len(test_df):,} baris  (Hoaks: {test_df['label'].sum():,} | Fakta: {len(test_df) - test_df['label'].sum():,})")
print(f"TOTAL DATASET    : {len(df_clean_master):,} BARIS (50:50 SEIMBANG)")
print("=" * 75)

🚀 FASE 2: PEMBERSIHAN MUTLAK & STRATIFIED SPLIT 70/15/15
Total Master Dataset Awal    : 14,276 baris
--> Mencabut kata 'imposter', 'periksa fakta', dan 'faktanya'...
Hasil Verifikasi Ulang Kebocoran : 0 baris bocor!
✅ STATUS: MASTER DATASET RESMI STERIL MUTLAK 100% (0 BOCOR)!

---------------------------------------------------------------------------
📦 MEMBAGI MENJADI 3 SUBSET RESMI (70% / 15% / 15%)...
🏆 BERHASIL! TIGA FILE RESMI TERSIMPAN DI data/processed/ :
• train.csv (70%) : 9,993 baris (Hoaks: 4,997 | Fakta: 4,996)
• val.csv   (15%) : 2,141 baris  (Hoaks: 1,070 | Fakta: 1,071)
• test.csv  (15%) : 2,142 baris  (Hoaks: 1,071 | Fakta: 1,071)
TOTAL DATASET    : 14,276 BARIS (50:50 SEIMBANG)


# HANYA MEMBACA (TIDAK MENGUBAH / TIDAK MERUSAK DATA) "Aturan Berhenti" (The Stopping Rule)

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import os

PROCESSED_DIR = "../data/processed" if os.path.exists("../data/processed") else "data/processed"

tr = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
va = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
te = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

print("=" * 70)
print("🔬 AUDIT 5 POIN AKHIR: MEMETAKAN KARAKTERISTIK DATASET PROCESSED")
print("=" * 70)

# Cek Jumlah baris rill
print(f"Baris Train: {len(tr):,} | Val: {len(va):,} | Test: {len(te):,} | TOTAL: {len(tr)+len(va)+len(te):,}")

# 1. BASELINE HANYA DARI TOPIK (Menguji apakah model bisa curang dari topik saja)
tmap = tr.groupby('sub_topic')['label'].agg(lambda s: int(s.mean() >= .5))
acc_topik = accuracy_score(te['label'], te['sub_topic'].map(tmap))
print(f"\n[1] Akurasi Prediksi MURNI DARI TOPIK SAJA : {acc_topik * 100:.2f}%")
print("    (Jika mendekati 75-79%, bias korelasi topik terbukti ada dan wajib ditulis di Bab 5)")

# 2. TF-IDF PADA SPLIT FINAL RIIL
vec = TfidfVectorizer(ngram_range=(1,2), min_df=3, sublinear_tf=True)
clf = LogisticRegression(max_iter=2000, random_state=42).fit(vec.fit_transform(tr['text']), tr['label'])
p = clf.predict(vec.transform(te['text']))
acc_tfidf = accuracy_score(te['label'], p)
f1_tfidf = f1_score(te['label'], p, average='macro')
print(f"\n[2] Akurasi Baseline Linear TF-IDF       : {acc_tfidf * 100:.2f}%")
print(f"    F1-Macro Baseline TF-IDF             : {f1_tfidf * 100:.2f}%")

# 3. AKURASI BASELINE PER SUB-TOPIK & LABEL DI DATA TEST
te_eval = te.copy()
te_eval['ok'] = (p == te_eval['label'])
print("\n[3] Performa Baseline per Sub-Topik & Label di Test Set:")
print(te_eval.groupby(['sub_topic','label'])['ok'].agg(['mean','size']).round(3))

# 4. KEBOCORAN NEAR-DUPLICATE ANTAR SPLIT (TRAIN VS TEST)
key = lambda d: d['text'].str[:80].str.lower().str.strip()
irisan = len(set(key(tr)) & set(key(te)))
print(f"\n[4] Kebocoran Irisan Train vs Test (80 Karakter Pertama) : {irisan} baris")
if irisan == 0:
    print("    ✓ Sempurna: Test set 100% independen dari Train set!")
else:
    print(f"    ⚠️ Ada {irisan} naskah mirip yang lolos antara train dan test.")

# 5. FREKUENSI SISA KATA ARTEFAK PER KELAS
print("\n[5] Frekuensi Kemunculan Kata Sisa per Kelas di Train Set:")
for w in [r'\bkategori\b', r'\bcontent\b', r'\btidak benar\b', r'\bbukan\b']:
    res = tr.groupby('label')['text'].apply(lambda s: s.str.contains(w, case=False, regex=True).mean()).round(4).to_dict()
    print(f"    • Pola '{w:<16}': {res} (0=Fakta, 1=Hoaks)")

print("=" * 70)


🔬 AUDIT 5 POIN AKHIR: MEMETAKAN KARAKTERISTIK DATASET PROCESSED
Baris Train: 9,993 | Val: 2,141 | Test: 2,142 | TOTAL: 14,276

[1] Akurasi Prediksi MURNI DARI TOPIK SAJA : 78.80%
    (Jika mendekati 75-79%, bias korelasi topik terbukti ada dan wajib ditulis di Bab 5)

[2] Akurasi Baseline Linear TF-IDF       : 93.79%
    F1-Macro Baseline TF-IDF             : 93.79%

[3] Performa Baseline per Sub-Topik & Label di Test Set:
                  mean  size
sub_topic label             
hukum     0      0.588    17
          1      0.976   165
kebijakan 0      0.842   184
          1      0.970   333
pemilu    0      0.972   799
          1      0.780   182
tokoh     0      0.845    71
          1      0.974   391

[4] Kebocoran Irisan Train vs Test (80 Karakter Pertama) : 0 baris
    ✓ Sempurna: Test set 100% independen dari Train set!

[5] Frekuensi Kemunculan Kata Sisa per Kelas di Train Set:
    • Pola '\bkategori\b    ': {0: 0.0006, 1: 0.043} (0=Fakta, 1=Hoaks)
    • Pola '\bcontent\b   